#### setup

In [ ]:
# make library code importable
import sys
from pathlib import Path
ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT / "library"))

# generic imports
import os
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

# library imports
from data_utils import *
from models import *
from training import *
from eval_utils import *

In [ ]:
# set device and seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### data

In [ ]:
# set dataset parameters
dataset = 'synthetic'
train_size = 1000

In [ ]:
# set confounders
confounders = ['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9']
input_dim = len(confounders)

In [ ]:
# read
data_path = ROOT / "data" / "datasets" / f"{dataset}.csv"
df = pd.read_csv(data_path, index_col=0)

#### helpers

In [ ]:
def load_checkpoint(model_cls, path, input_dim, device, hidden_dims=(128, 64)):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {path}")

    last_error = None

    for hidden_dim in hidden_dims:
        try:
            model = model_cls(input_dim=input_dim, hidden_dim=hidden_dim).to(device)
            model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
            model.eval()
            return model
        except RuntimeError as err:
            last_error = err

    raise RuntimeError(f"Could not load checkpoint with hidden dims {hidden_dims}: {path}") from last_error

In [ ]:
def load_models(seed, dataset, train_size, confounders, device):
    input_dim = len(confounders)

    ranker_dir = ROOT / "experiments" / "rankers" / "chkpts" / dataset / f"size_{train_size}" / f"seed_{seed}"
    pointwise_dir = ROOT / "experiments" / "pointwise" / "chkpts" / dataset / f"size_{train_size}" / f"seed_{seed}"
    nuisance_dir = ROOT / "experiments" / "nuisances" / "chkpts" / dataset / f"size_{train_size}" / f"seed_{seed}"

    orth_model = load_checkpoint(
        ClassificationHead,
        ranker_dir / "orthogonal.pt",
        input_dim,
        device,
    )

    pi_model = load_checkpoint(
        ClassificationHead,
        ranker_dir / "plug_in.pt",
        input_dim,
        device,
    )

    dr_model = load_checkpoint(
        RegressionHead,
        pointwise_dir / "cate_model.pt",
        input_dim,
        device,
    )

    m0_model = load_checkpoint(
        RegressionHead,
        nuisance_dir / "mu0_model.pt",
        input_dim,
        device,
    )

    m1_model = load_checkpoint(
        RegressionHead,
        nuisance_dir / "mu1_model.pt",
        input_dim,
        device,
    )

    return orth_model, pi_model, dr_model, m0_model, m1_model

#### evaluation

In [ ]:
# init collector
all_metrics = []

# loop over seeds
for seed in range(5):

    # track progress
    print(f" -> Seed {seed}, size {train_size}")
    set_seed(seed)

    # get testing data
    _, _, _, _, test_df = make_splits(df=df, train_size=train_size, seed=seed)
    test_loader = DataLoader(EvalDataset(test_df, confounders), batch_size=1024, shuffle=False)

    # load models
    rank_learner, pi_model, dr_model, m0_model, m1_model = load_models(seed=seed, dataset=dataset, train_size=train_size, confounders=confounders, device=device)

    # get predictions
    df_eval = get_estimates_all(rank_learner, pi_model, dr_model, m0_model, m1_model, test_loader, device)

    # compute and store metrics
    df_metrics = compute_metrics_all(df_eval)
    df_metrics["seed"] = seed
    df_metrics["train_size"] = train_size
    all_metrics.append(df_metrics)

# summarize
df_all = pd.concat(all_metrics, ignore_index=True)
metrics_to_avg = ["autoc", "policy_value"]
df_summary = (df_all.groupby(["model", "train_size"])[metrics_to_avg].agg(["mean", "std"]).reset_index())